# 最適作物マップ（長期的供給係数版）

- **最適作物**: カロリーベース（kcal/ha）で単収が最も高い穀物（Maize / Rice / Wheat）
- **カロリー係数**: Maize 3960, Rice 3840, Wheat 3750（FAO 14%水分）
- **長期的供給係数**（連作障害・`continuous_cropping_supply_coefficients.md`）: **effective_y = y × 係数**
  - **コーン**: `effective_y = y * 0.85`
  - **小麦**: `effective_y = y * 0.86`
  - **米**: `effective_y = y * 0.8`
  - 最適作物は `effective_kcal = yield × kcal/kg × 供給係数` で比較
- **5シナリオ**: 2013（2011–2013平均）、2100 SSP3-7.0 平均・95%CI下限、2100 SSP2-4.5 平均・95%CI下限
- **対象**: 予測精度上位65か国。地図は ISO-3。

In [1]:
import pandas as pd
import numpy as np
import os
import plotly.express as px

base_dir = r'C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット'
os.chdir(base_dir)

reg = pd.read_excel(os.path.join(base_dir, 'regression_df_20260112_233447.xlsx'))
pred = pd.read_csv(os.path.join(base_dir, 'predict_2100_bootstrap_results.csv'))

ITEMS = ['Maize (corn)', 'Rice', 'Wheat']
KCAL_PER_KG = {'Maize (corn)': 3960, 'Rice': 3840, 'Wheat': 3750}
# 長期的供給係数: effective_y = y * 係数（連作障害）
# コーン 0.85, 小麦 0.86, 米 0.8
SUPPLY_COEF = {'Maize (corn)': 0.85, 'Rice': 0.8, 'Wheat': 0.86}

COUNTRY_65 = {
    'Greece', 'Zambia', 'Italy', 'Thailand', 'Canada', 'Peru', 'United States of America',
    'Dominican Republic', 'Republic of Korea', 'Kazakhstan', 'El Salvador', 'Suriname', 'Eswatini',
    'Oman', 'Austria', 'Kuwait', 'Costa Rica', 'Argentina', 'Namibia', 'Chile', 'Spain', 'Panama',
    'Ecuador', 'Myanmar', 'Belgium', 'Qatar', 'Uruguay', 'Guyana', 'Morocco', 'Honduras', 'Australia',
    'Mauritania', 'Colombia', 'France', 'Japan', 'Haiti', 'New Zealand', 'Fiji', 'Niger', 'Switzerland',
    'Poland', 'Russian Federation', 'Somalia', 'Guatemala', 'Comoros', 'Mexico', 'Portugal', 'Armenia',
    'Türkiye', 'Slovenia', 'China', 'Syrian Arab Republic', 'Trinidad and Tobago', 'Ghana', 'North Macedonia',
    'Belarus', 'Togo', 'Rwanda', 'Libya', 'Guinea-Bissau', 'Burundi', 'Israel', 'Germany', 'Bangladesh', 'Nicaragua'
}

NAME_MAP = {
    'United States of America': 'United States', 'Republic of Korea': 'South Korea',
    'Russian Federation': 'Russia', 'Syrian Arab Republic': 'Syria', 'Türkiye': 'Turkey',
    'Viet Nam': 'Vietnam', "Côte d'Ivoire": 'Ivory Coast', 'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
}

ISO3_MAP = {
    'Greece': 'GRC', 'Zambia': 'ZMB', 'Italy': 'ITA', 'Thailand': 'THA', 'Canada': 'CAN', 'Peru': 'PER',
    'United States of America': 'USA', 'Dominican Republic': 'DOM', 'Republic of Korea': 'KOR', 'Kazakhstan': 'KAZ',
    'El Salvador': 'SLV', 'Suriname': 'SUR', 'Eswatini': 'SWZ', 'Oman': 'OMN', 'Austria': 'AUT', 'Kuwait': 'KWT',
    'Costa Rica': 'CRI', 'Argentina': 'ARG', 'Namibia': 'NAM', 'Chile': 'CHL', 'Spain': 'ESP', 'Panama': 'PAN',
    'Ecuador': 'ECU', 'Myanmar': 'MMR', 'Belgium': 'BEL', 'Qatar': 'QAT', 'Uruguay': 'URY', 'Guyana': 'GUY',
    'Morocco': 'MAR', 'Honduras': 'HND', 'Australia': 'AUS', 'Mauritania': 'MRT', 'Colombia': 'COL', 'France': 'FRA',
    'Japan': 'JPN', 'Haiti': 'HTI', 'New Zealand': 'NZL', 'Fiji': 'FJI', 'Niger': 'NER', 'Switzerland': 'CHE',
    'Poland': 'POL', 'Russian Federation': 'RUS', 'Somalia': 'SOM', 'Guatemala': 'GTM', 'Comoros': 'COM',
    'Mexico': 'MEX', 'Portugal': 'PRT', 'Armenia': 'ARM', 'Türkiye': 'TUR', 'Slovenia': 'SVN', 'China': 'CHN',
    'Syrian Arab Republic': 'SYR', 'Trinidad and Tobago': 'TTO', 'Ghana': 'GHA', 'North Macedonia': 'MKD',
    'Belarus': 'BLR', 'Togo': 'TGO', 'Rwanda': 'RWA', 'Libya': 'LBY', 'Guinea-Bissau': 'GNB', 'Burundi': 'BDI',
    'Israel': 'ISR', 'Germany': 'DEU', 'Bangladesh': 'BGD', 'Nicaragua': 'NIC',
}

def yield_to_kcal(df, yield_col):
    out = df.copy()
    out['kcal_per_ha'] = out[yield_col] * out['Item'].map(KCAL_PER_KG)
    out['effective_kcal'] = out['kcal_per_ha'] * out['Item'].map(SUPPLY_COEF)
    return out

def optimal_crop_per_country(df):
    idx = df.groupby('Area')['effective_kcal'].idxmax()
    best = df.loc[idx, ['Area', 'Item', 'effective_kcal', 'kcal_per_ha']].copy()
    best = best.rename(columns={'Item': 'optimal_crop'})
    return best[['Area', 'optimal_crop', 'effective_kcal', 'kcal_per_ha']]

print('供給係数: Maize 0.85, Wheat 0.86, Rice 0.8')
print('regression_df:', reg.shape, '| predict:', pred.shape)

供給係数: Maize 0.85, Wheat 0.86, Rice 0.8
regression_df: (16388, 8) | predict: (936, 5)


In [2]:
y13 = reg[(reg['Year'].between(2011, 2013)) & (reg['Item'].isin(ITEMS))][['Area', 'Item', 'Yield']].copy()
y13 = y13[y13['Area'].isin(COUNTRY_65)]
y13 = y13.groupby(['Area', 'Item'], as_index=False)['Yield'].mean()
y13 = y13.rename(columns={'Yield': 'yield_kg'})
y13 = yield_to_kcal(y13, 'yield_kg')
opt_2013 = optimal_crop_per_country(y13)
opt_2013['scenario'] = '2013 (2011–2013平均)'

p = pred[pred['Area'].isin(COUNTRY_65)].copy()
p370m = p[p['scenario'] == 'SSP370'][['Area', 'Item', 'mean_yield']].rename(columns={'mean_yield': 'y'})
p370m = yield_to_kcal(p370m, 'y')
opt_370m = optimal_crop_per_country(p370m)
opt_370m['scenario'] = '2100 SSP3-7.0 平均'

p370c = p[p['scenario'] == 'SSP370'][['Area', 'Item', 'ci95_lower']].rename(columns={'ci95_lower': 'y'})
p370c = yield_to_kcal(p370c, 'y')
opt_370c = optimal_crop_per_country(p370c)
opt_370c['scenario'] = '2100 SSP3-7.0 95%CI下限'

p245m = p[p['scenario'] == 'SSP245'][['Area', 'Item', 'mean_yield']].rename(columns={'mean_yield': 'y'})
p245m = yield_to_kcal(p245m, 'y')
opt_245m = optimal_crop_per_country(p245m)
opt_245m['scenario'] = '2100 SSP2-4.5 平均'

p245c = p[p['scenario'] == 'SSP245'][['Area', 'Item', 'ci95_lower']].rename(columns={'ci95_lower': 'y'})
p245c = yield_to_kcal(p245c, 'y')
opt_245c = optimal_crop_per_country(p245c)
opt_245c['scenario'] = '2100 SSP2-4.5 95%CI下限'

all_opt = pd.concat([opt_2013, opt_370m, opt_370c, opt_245m, opt_245c], ignore_index=True)
all_opt['Country'] = all_opt['Area'].replace(NAME_MAP)
all_opt['ISO3'] = all_opt['Area'].map(ISO3_MAP)
order = ['2013 (2011–2013平均)', '2100 SSP3-7.0 平均', '2100 SSP3-7.0 95%CI下限', '2100 SSP2-4.5 平均', '2100 SSP2-4.5 95%CI下限']
all_opt['scenario'] = pd.Categorical(all_opt['scenario'], categories=order, ordered=True)

print('=== シナリオ別 国数 ===')
for s in order:
    n = (all_opt['scenario'] == s).sum()
    print(f'  {s}: {n}か国')
print()
print('=== シナリオ別 最適作物の内訳（供給係数 コーン0.85, 小麦0.86, 米0.8） ===')
print(all_opt.groupby(['scenario', 'optimal_crop'], observed=True).size().unstack(fill_value=0))
print(all_opt.head(10))

=== シナリオ別 国数 ===
  2013 (2011–2013平均): 65か国
  2100 SSP3-7.0 平均: 65か国
  2100 SSP3-7.0 95%CI下限: 65か国
  2100 SSP2-4.5 平均: 65か国
  2100 SSP2-4.5 95%CI下限: 65か国

=== シナリオ別 最適作物の内訳（供給係数 コーン0.85, 小麦0.86, 米0.8） ===
optimal_crop           Maize (corn)  Rice  Wheat
scenario                                        
2013 (2011–2013平均)               31    31      3
2100 SSP3-7.0 平均                 30    33      2
2100 SSP3-7.0 95%CI下限            29    34      2
2100 SSP2-4.5 平均                 31    32      2
2100 SSP2-4.5 95%CI下限            30    33      2
         Area  optimal_crop  effective_kcal  kcal_per_ha            scenario  \
0   Argentina  Maize (corn)      20968609.2   24668952.0  2013 (2011–2013平均)   
1     Armenia  Maize (corn)      20598349.2   24233352.0  2013 (2011–2013平均)   
2   Australia          Rice      29359718.4   36699648.0  2013 (2011–2013平均)   
3     Austria  Maize (corn)      33793966.8   39757608.0  2013 (2011–2013平均)   
4  Bangladesh  Maize (corn)      21688596.6   255159

### 2013との比較

各2100シナリオについて、2013（2011–2013平均）と同じ最適作物の国数・変更した国数、および変更内訳（2013→2100）を表示する。

In [3]:
base = opt_2013[['Area', 'optimal_crop']].rename(columns={'optimal_crop': 'opt_2013'})
future_scenarios = ['2100 SSP3-7.0 平均', '2100 SSP3-7.0 95%CI下限', '2100 SSP2-4.5 平均', '2100 SSP2-4.5 95%CI下限']

print('=== 2013との比較 ===')
for s in future_scenarios:
    fut = all_opt[all_opt['scenario'] == s][['Area', 'optimal_crop']].copy()
    fut = fut.rename(columns={'optimal_crop': 'opt_2100'})
    m = base.merge(fut, on='Area')
    same = (m['opt_2013'] == m['opt_2100']).sum()
    chg = len(m) - same
    print(f'\n{s}:')
    print(f'  2013と同じ最適作物: {same}か国')
    print(f'  2013から変更: {chg}か国')
    chgd = m[m['opt_2013'] != m['opt_2100']]
    if len(chgd) > 0:
        trans = chgd.groupby(['opt_2013', 'opt_2100']).size().reset_index(name='n')
        print('  変更内訳（2013 → 2100）:')
        for _, r in trans.iterrows():
            print(f'    {r["opt_2013"]} → {r["opt_2100"]}: {int(r["n"])}か国')
            countries = chgd[(chgd['opt_2013'] == r['opt_2013']) & (chgd['opt_2100'] == r['opt_2100'])]['Area'].sort_values().tolist()
            print(f'      → {", ".join(countries)}')
        print('  変更した国（一覧）: ' + ', '.join(chgd['Area'].sort_values().tolist()))

=== 2013との比較 ===

2100 SSP3-7.0 平均:
  2013と同じ最適作物: 59か国
  2013から変更: 6か国
  変更内訳（2013 → 2100）:
    Maize (corn) → Rice: 3か国
      → Myanmar, Syrian Arab Republic, Türkiye
    Rice → Maize (corn): 2か国
      → North Macedonia, Russian Federation
    Wheat → Rice: 1か国
      → Namibia
  変更した国（一覧）: Myanmar, Namibia, North Macedonia, Russian Federation, Syrian Arab Republic, Türkiye

2100 SSP3-7.0 95%CI下限:
  2013と同じ最適作物: 60か国
  2013から変更: 5か国
  変更内訳（2013 → 2100）:
    Maize (corn) → Rice: 3か国
      → Myanmar, Syrian Arab Republic, Türkiye
    Rice → Maize (corn): 1か国
      → Russian Federation
    Wheat → Rice: 1か国
      → Namibia
  変更した国（一覧）: Myanmar, Namibia, Russian Federation, Syrian Arab Republic, Türkiye

2100 SSP2-4.5 平均:
  2013と同じ最適作物: 60か国
  2013から変更: 5か国
  変更内訳（2013 → 2100）:
    Maize (corn) → Rice: 2か国
      → Myanmar, Syrian Arab Republic
    Rice → Maize (corn): 2か国
      → North Macedonia, Russian Federation
    Wheat → Rice: 1か国
      → Namibia
  変更した国（一覧）: Myanmar, Namibia, North

### 最適作物の変更有無マップ（赤=変更・白=不変・グレー=対象外）

各2100シナリオについて、2013→2100で最適作物が**変わった国を赤**、**変わらなかった国を白**、**対象外（65か国以外）をグレー**で表示する。

In [12]:
# 最適作物が変わった国=赤、変わらなかった国=白、対象外=グレー（地図の陸色）
for s in future_scenarios:
    fut = all_opt[all_opt['scenario'] == s][['Area', 'optimal_crop']].rename(columns={'optimal_crop': 'opt_2100'})
    m = base.merge(fut, on='Area')
    m['changed'] = (m['opt_2013'] != m['opt_2100']).astype(int)  # 1=変更, 0=不変
    m['ISO3'] = m['Area'].map(ISO3_MAP)
    fig = px.choropleth(
        m,
        locations='ISO3',
        locationmode='ISO-3',
        color='changed',
        color_continuous_scale=[[0, '#ffffff'], [1, '#e74c3c']],
        range_color=[0, 1],
        hover_name='Area',
        hover_data={'opt_2013': True, 'opt_2100': True, 'changed': False, 'ISO3': False},
        title=f'最適作物の変更有無（2013→{s}） — 赤=変更・白=不変・グレー=対象外'
    )
    fig.update_layout(
        margin=dict(l=0, r=0, t=50, b=0),
        height=420,
        geo=dict(showframe=False, showcoastlines=True, landcolor='#95a5a6'),
        coloraxis_colorbar=dict(
            title='',
            tickvals=[0, 1],
            ticktext=['不変', '変更'],
            len=0.35,
            y=0.85
        )
    )
    fig.show()
print('4 maps (最適作物の変更有無: 赤/白/グレー) done.')

4 maps (最適作物の変更有無: 赤/白/グレー) done.


### 2013 vs 2100 最適作物カロリー比較（パーセント変化・グラデーション）

各2100シナリオについて、2013の最適作物の effective_kcal との**パーセント変化** `(kcal_2100 − kcal_2013) / kcal_2013 × 100` を算出。増加側はオレンジ、減少側は青のグラデーションで地図に表示する。**40%以上は最大の濃さ**（同じ色）で塗る。

In [4]:
base_kcal = opt_2013[['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2013'})
future_scenarios = ['2100 SSP3-7.0 平均', '2100 SSP3-7.0 95%CI下限', '2100 SSP2-4.5 平均', '2100 SSP2-4.5 95%CI下限']
# 青(減少) → 白(0) → オレンジ(増加) のグラデーション
scale_div = [[0, '#3498db'], [0.5, '#ecf0f1'], [1, '#e67e22']]

for s in future_scenarios:
    fut = all_opt[all_opt['scenario'] == s][['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2100'})
    m = base_kcal.merge(fut, on='Area')
    m['diff'] = m['kcal_2100'] - m['kcal_2013']
    m['pct_change'] = np.where(m['kcal_2013'] > 0, m['diff'] / m['kcal_2013'] * 100, np.nan)
    m['ISO3'] = m['Area'].map(ISO3_MAP)
    d = m.dropna(subset=['ISO3', 'pct_change'])
    fig = px.choropleth(
        d,
        locations='ISO3',
        locationmode='ISO-3',
        color='pct_change',
        color_continuous_scale=scale_div,
        color_continuous_midpoint=0,
        range_color=[-40, 40],
        hover_name='Area',
        hover_data={'pct_change': ':.1f', 'kcal_2013': ':,.0f', 'kcal_2100': ':,.0f', 'diff': ':,.0f', 'ISO3': False},
        title=f'2013 vs {s} 最適作物カロリー変化（%）'
    )
    fig.update_layout(
        margin=dict(l=0, r=0, t=50, b=0),
        height=420,
        geo=dict(showframe=False, showcoastlines=True, landcolor='#95a5a6'),
        coloraxis_colorbar_title='2013比（%）'
    )
    fig.show()
print('4 maps (2013 vs 2100 パーセント変化・グラデーション) done.')

4 maps (2013 vs 2100 パーセント変化・グラデーション) done.


### まとめ: 2013 vs 2100 最適作物カロリー変化（4シナリオ一覧）

上記4枚の地図を1つの図にまとめた版（2×2）。

In [5]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

base_kcal = opt_2013[['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2013'})
future_scenarios = ['2100 SSP3-7.0 平均', '2100 SSP3-7.0 95%CI下限', '2100 SSP2-4.5 平均', '2100 SSP2-4.5 95%CI下限']
scale_div = [[0, '#3498db'], [0.5, '#ecf0f1'], [1, '#e67e22']]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'2013 vs {s}' for s in future_scenarios],
    specs=[[{"type": "geo"}, {"type": "geo"}], [{"type": "geo"}, {"type": "geo"}]],
    vertical_spacing=0.08, horizontal_spacing=0.02,
)
for idx, s in enumerate(future_scenarios):
    fut = all_opt[all_opt['scenario'] == s][['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2100'})
    m = base_kcal.merge(fut, on='Area')
    m['pct_change'] = np.where(m['kcal_2013'] > 0, (m['kcal_2100'] - m['kcal_2013']) / m['kcal_2013'] * 100, np.nan)
    m['ISO3'] = m['Area'].map(ISO3_MAP)
    d = m.dropna(subset=['ISO3', 'pct_change'])
    row, col = idx // 2 + 1, idx % 2 + 1
    fig.add_trace(
        go.Choropleth(
            locations=d['ISO3'],
            z=d['pct_change'],
            locationmode='ISO-3',
            colorscale=scale_div,
            zmin=-40, zmax=40,
            hovertext=[f"{a}<br>{v:.1f}%" for a, v in zip(d['Area'], d['pct_change'])],
            hoverinfo='text',
            showscale=(idx == 0),
            colorbar=dict(title='2013比（%）', len=0.4, y=0.95) if idx == 0 else None,
        ),
        row=row, col=col,
    )
for r in range(1, 3):
    for c in range(1, 3):
        fig.update_geos(showframe=False, showcoastlines=True, landcolor='#95a5a6', row=r, col=c)
fig.update_layout(
    height=700,
    margin=dict(l=0, r=0, t=60, b=0),
    title_text='2013 vs 2100 最適作物カロリー変化（%）— 4シナリオ一覧',
)
fig.show()

### 最適作物カロリーの変化の表まとめ

2013（2011–2013平均）の最適作物の effective_kcal と各2100シナリオの最適作物の effective_kcal を比較し、**パーセント変化** `(kcal_2100 − kcal_2013) / kcal_2013 × 100` をシナリオ別・国別に表にまとめる。

In [6]:
base_kcal = opt_2013[['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2013'})
future_scenarios = ['2100 SSP3-7.0 平均', '2100 SSP3-7.0 95%CI下限', '2100 SSP2-4.5 平均', '2100 SSP2-4.5 95%CI下限']

# 国×シナリオのパーセント変化を計算
rows = []
for s in future_scenarios:
    fut = all_opt[all_opt['scenario'] == s][['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2100'})
    m = base_kcal.merge(fut, on='Area')
    m['diff'] = m['kcal_2100'] - m['kcal_2013']
    m['pct_change'] = np.where(m['kcal_2013'] > 0, m['diff'] / m['kcal_2013'] * 100, np.nan)
    m['scenario'] = s
    rows.append(m[['Area', 'scenario', 'kcal_2013', 'kcal_2100', 'diff', 'pct_change']])
pct_df = pd.concat(rows, ignore_index=True)

# 表1: シナリオ別サマリ（平均・中央値・最小・最大・増加/減少国数）
summary = pct_df.groupby('scenario').agg(
    平均変化率_=('pct_change', 'mean'),
    中央値_=('pct_change', 'median'),
    最小_=('pct_change', 'min'),
    最大_=('pct_change', 'max'),
    増加国数=('pct_change', lambda x: (x > 0).sum()),
    減少国数=('pct_change', lambda x: (x < 0).sum()),
    変化なし=('pct_change', lambda x: (x == 0).sum()),
).round(2)
summary.columns = ['平均変化率（%）', '中央値（%）', '最小（%）', '最大（%）', '増加国数', '減少国数', '変化なし']
print('=== シナリオ別 最適作物カロリー変化のサマリ（2013比） ===')
display(summary)

# 表の10か国（中国・米国・インド・ブラジル・ロシア・アルゼンチン・インドネシア・バングラデシュ・カナダ・パキスタン）の平均（モデルにあった国のみ）
TOP10_AREAS = {'China', 'United States of America', 'India', 'Brazil', 'Russian Federation', 'Argentina', 'Indonesia', 'Bangladesh', 'Canada', 'Pakistan'}
in_model = TOP10_AREAS & set(pct_df['Area'].unique())
missing = TOP10_AREAS - in_model
if missing:
    print('\n※ モデルに含まれていない国（対象外）: ' + ', '.join(sorted(missing)))
pct_top10 = pct_df[pct_df['Area'].isin(in_model)]
if len(in_model) > 0:
    summary_top10 = pct_top10.groupby('scenario').agg(
        平均変化率_=('pct_change', 'mean'),
        中央値_=('pct_change', 'median'),
        最小_=('pct_change', 'min'),
        最大_=('pct_change', 'max'),
        対象国数=('pct_change', 'count'),
    ).round(2)
    summary_top10.columns = ['平均変化率（%）', '中央値（%）', '最小（%）', '最大（%）', '対象国数']
    print('=== 表の10か国（モデルにあった国のみ）のシナリオ別 最適作物カロリー変化のサマリ（2013比） ===')
    print('対象国: ' + ', '.join(sorted(in_model)))
    display(summary_top10)

# 表2: 国×シナリオ パーセント変化（ピボット）
pivot_pct = pct_df.pivot(index='Area', columns='scenario', values='pct_change').round(1)
pivot_pct = pivot_pct[future_scenarios]
print('\n=== 国別・シナリオ別 最適作物カロリー変化（%、2013比） ===')
display(pivot_pct)

# 表3: 国別 2013のカロリーと各シナリオのカロリー・変化率（縦長）
wide = pct_df.pivot(index='Area', columns='scenario', values=['kcal_2013', 'kcal_2100', 'pct_change'])
# kcal_2013 は全シナリオで同じなので1列でよい
table_cal = opt_2013[['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2013'})
for s in future_scenarios:
    sub = pct_df[pct_df['scenario'] == s][['Area', 'kcal_2100', 'pct_change']]
    sub = sub.rename(columns={'kcal_2100': f'kcal_2100_{s}', 'pct_change': f'pct_{s}'})
    table_cal = table_cal.merge(sub, on='Area', how='left')
table_cal = table_cal.round({'kcal_2013': 0})
for s in future_scenarios:
    table_cal[f'pct_{s}'] = table_cal[f'pct_{s}'].round(1)
print('\n=== 国別 2013 effective_kcal と 2100 カロリー・変化率（参考） ===')
display(table_cal.head(15))

# 表4: 変化率の上位から全か国を並べた表（2100 SSP3-7.0 95%CI下限のみ・全行表示）
s = '2100 SSP3-7.0 95%CI下限'
t = pct_df[pct_df['scenario'] == s][['Area', 'pct_change', 'kcal_2013', 'kcal_2100']].copy()
t = t.sort_values('pct_change', ascending=False).reset_index(drop=True)
t.insert(0, '順位', np.arange(1, len(t) + 1))
t['変化率（%）'] = t['pct_change'].round(1)
t['kcal_2013'] = t['kcal_2013'].round(0)
t['kcal_2100'] = t['kcal_2100'].round(0)
display_df = t[['順位', 'Area', '変化率（%）', 'kcal_2013', 'kcal_2100']].rename(columns={'Area': '国'})
print('=== 最適作物カロリー変化率 上位から全か国（2013比%、高い→低い） --- 2100 SSP3-7.0 95%CI下限 ===')
pd.set_option('display.max_rows', None)
display(display_df)
pd.reset_option('display.max_rows')

=== シナリオ別 最適作物カロリー変化のサマリ（2013比） ===


,平均変化率（%）,中央値（%）,最小（%）,最大（%）,増加国数,減少国数,変化なし
scenario,,,,,,,
2100 SSP2-4.5 95%CI下限,-17.40,-16.76,-50.60,30.17,6,59,0
2100 SSP2-4.5 平均,-1.15,-3.50,-35.72,176.32,25,40,0
2100 SSP3-7.0 95%CI下限,-19.37,-20.15,-47.96,38.85,6,59,0
2100 SSP3-7.0 平均,-1.51,-7.75,-35.89,233.35,26,39,0



※ モデルに含まれていない国（対象外）: Brazil, India, Indonesia, Pakistan
=== 表の10か国（モデルにあった国のみ）のシナリオ別 最適作物カロリー変化のサマリ（2013比） ===
対象国: Argentina, Bangladesh, Canada, China, Russian Federation, United States of America


,平均変化率（%）,中央値（%）,最小（%）,最大（%）,対象国数
scenario,,,,,
2100 SSP2-4.5 95%CI下限,-25.22,-23.07,-46.56,-11.24,6
2100 SSP2-4.5 平均,-10.99,-9.87,-24.73,-0.81,6
2100 SSP3-7.0 95%CI下限,-22.97,-25.70,-36.38,-7.10,6
2100 SSP3-7.0 平均,-9.93,-12.00,-17.56,1.55,6



=== 国別・シナリオ別 最適作物カロリー変化（%、2013比） ===


scenario,2100 SSP3-7.0 平均,2100 SSP3-7.0 95%CI下限,2100 SSP2-4.5 平均,2100 SSP2-4.5 95%CI下限
Area,,,,
Argentina,-7.8,-22.1,-3.1,-15.7
Armenia,-11.2,-19.3,-9.2,-16.7
Australia,-7.7,-22.3,-11.5,-26.3
Austria,-12.7,-27.7,-10.5,-24.7
Bangladesh,-17.6,-36.4,-24.7,-46.6
...,...,...,...,...
Trinidad and Tobago,5.6,-6.4,3.7,-6.7
Türkiye,-24.8,-39.6,-10.3,-26.3
United States of America,-16.2,-32.2,-16.2,-32.5



=== 国別 2013 effective_kcal と 2100 カロリー・変化率（参考） ===


,Area,kcal_2013,kcal_2100_2100 SSP3-7.0 平均,pct_2100 SSP3-7.0 平均,kcal_2100_2100 SSP3-7.0 95%CI下限,pct_2100 SSP3-7.0 95%CI下限,kcal_2100_2100 SSP2-4.5 平均,pct_2100 SSP2-4.5 平均,kcal_2100_2100 SSP2-4.5 95%CI下限,pct_2100 SSP2-4.5 95%CI下限
0,Argentina,20968609.0,1.933597e+07,-7.8,1.632474e+07,-22.1,2.032830e+07,-3.1,1.766929e+07,-15.7
1,Armenia,20598349.0,1.830135e+07,-11.2,1.662534e+07,-19.3,1.870254e+07,-9.2,1.715665e+07,-16.7
2,Australia,29359718.0,2.708560e+07,-7.7,2.281971e+07,-22.3,2.598958e+07,-11.5,2.164650e+07,-26.3
3,Austria,33793967.0,2.951734e+07,-12.7,2.444476e+07,-27.7,3.024023e+07,-10.5,2.543914e+07,-24.7
4,Bangladesh,21688597.0,1.787915e+07,-17.6,1.379795e+07,-36.4,1.632583e+07,-24.7,1.159039e+07,-46.6
5,Belarus,19299859.0,1.694666e+07,-12.2,1.403183e+07,-27.3,1.955727e+07,1.3,1.606571e+07,-16.8
6,Belgium,37396036.0,3.618991e+07,-3.2,3.152489e+07,-15.7,3.852224e+07,3.0,3.460601e+07,-7.5
7,Burundi,6862746.0,8.398519e+06,22.4,7.213145e+06,5.1,8.467021e+06,23.4,7.195222e+06,4.8
8,Canada,31005572.0,2.564631e+07,-17.3,2.193795e+07,-29.2,2.555083e+07,-17.6,2.157722e+07,-30.4
9,Chile,37421281.0,2.732649e+07,-27.0,2.180372e+07,-41.7,3.310240e+07,-11.5,2.783961e+07,-25.6


=== 最適作物カロリー変化率 上位から全か国（2013比%、高い→低い） --- 2100 SSP3-7.0 95%CI下限 ===


,順位,国,変化率（%）,kcal_2013,kcal_2100
0,1,Libya,38.8,7124027.0,9891561.0
1,2,Syrian Arab Republic,11.0,14555482.0,16157132.0
2,3,Guinea-Bissau,8.0,5290189.0,5712503.0
3,4,Burundi,5.1,6862746.0,7213145.0
4,5,Comoros,4.9,6462608.0,6781043.0
5,6,Fiji,0.5,8693555.0,8733758.0
6,7,Qatar,-0.9,42108099.0,41738268.0
7,8,Republic of Korea,-1.6,20179354.0,19865861.0
8,9,Oman,-1.6,25715455.0,25302160.0
9,10,Togo,-2.8,6143488.0,5970036.0


### 最適作物カロリー変化のシナリオ別比較

各2100シナリオ（SSP3-7.0/SSP2-4.5 × 平均/95%CI下限）ごとの最適作物カロリー変化率（2013比%）を、シナリオ間で比較する。サマリの棒グラフと国別分布の箱ひげ図を表示。

In [7]:
# シナリオ別比較（pct_df は直前のセルで作成済み）
order_sc = ['2100 SSP3-7.0 平均', '2100 SSP3-7.0 95%CI下限', '2100 SSP2-4.5 平均', '2100 SSP2-4.5 95%CI下限']
pct_df['scenario'] = pd.Categorical(pct_df['scenario'], categories=order_sc, ordered=True)

# 1) シナリオ別 平均・中央値の棒グラフ
sum_plot = pct_df.groupby('scenario', observed=True).agg(
    平均=('pct_change', 'mean'),
    中央値=('pct_change', 'median'),
).reset_index()
sum_long = sum_plot.melt(id_vars='scenario', value_vars=['平均', '中央値'], var_name='指標', value_name='変化率（%）')

fig1 = px.bar(
    sum_long, x='scenario', y='変化率（%）', color='指標',
    barmode='group', title='最適作物カロリー変化率のシナリオ別比較（2013比%）',
    labels={'scenario': 'シナリオ'},
    color_discrete_map={'平均': '#e67e22', '中央値': '#3498db'},
)
fig1.update_layout(
    xaxis_tickangle=-25,
    height=400,
    legend_title='指標',
    yaxis_title='変化率（%）',
)
fig1.add_hline(y=0, line_dash='dot', line_color='gray')
fig1.show()

# 2) 国別変化率の分布（シナリオ別 箱ひげ図）
fig2 = px.box(
    pct_df, x='scenario', y='pct_change',
    title='国別 最適作物カロリー変化率の分布（シナリオ別・2013比%）',
    labels={'scenario': 'シナリオ', 'pct_change': '変化率（%）'},
)
fig2.update_layout(height=420, xaxis_tickangle=-25, showlegend=False)
fig2.add_hline(y=0, line_dash='dot', line_color='gray')
fig2.show()

print('シナリオ別比較（棒グラフ・箱ひげ図）表示完了。')

シナリオ別比較（棒グラフ・箱ひげ図）表示完了。


### 最適 vs 2番目 の差（%）— 0–20% グラデーション、20%超は濃色

最適作物の effective_kcal と2番目の作物の差を **(最適 − 2番目) / 2番目 × 100** で算出。0–20% をグラデーションで表示し、**20%超はすべて濃い色**（同じ）で塗る。

In [8]:
def best_vs_second(df):
    """Area, Item, effective_kcal の df → 最適・2番目・pct_diff"""
    rows = []
    for area, g in df.groupby('Area'):
        g = g.sort_values('effective_kcal', ascending=False)
        if len(g) < 2:
            continue
        r1, r2 = g.iloc[0], g.iloc[1]
        best_k, second_k = r1['effective_kcal'], r2['effective_kcal']
        if second_k <= 0:
            continue
        pct = (best_k - second_k) / second_k * 100
        rows.append({
            'Area': area,
            'optimal_crop': r1['Item'],
            'second_crop': r2['Item'],
            'best_kcal': best_k,
            'second_kcal': second_k,
            'pct_diff': pct,
        })
    return pd.DataFrame(rows)

g2013 = best_vs_second(y13)
g2013['scenario'] = '2013 (2011–2013平均)'
gaps = [g2013]
for _df, _s in [(p370m, '2100 SSP3-7.0 平均'), (p370c, '2100 SSP3-7.0 95%CI下限'),
                (p245m, '2100 SSP2-4.5 平均'), (p245c, '2100 SSP2-4.5 95%CI下限')]:
    g = best_vs_second(_df)
    g['scenario'] = _s
    gaps.append(g)
gap_all = pd.concat(gaps, ignore_index=True)
gap_all['ISO3'] = gap_all['Area'].map(ISO3_MAP)
# 0–20% グラデーション、20%超は濃色（range_color でクリップ）
scale_gap = [[0, '#ecf0f1'], [1, '#2c3e50']]

for s in order:
    d = gap_all[gap_all['scenario'] == s].dropna(subset=['ISO3'])
    fig = px.choropleth(
        d,
        locations='ISO3',
        locationmode='ISO-3',
        color='pct_diff',
        range_color=[0, 20],
        color_continuous_scale=scale_gap,
        hover_name='Area',
        hover_data={'optimal_crop': True, 'second_crop': True, 'pct_diff': ':.1f', 'best_kcal': ':,.0f', 'second_kcal': ':,.0f', 'ISO3': False},
        title=f'最適 vs 2番目 の差（%） — {s}  [0–20% グラデーション、20%超は濃色]'
    )
    fig.update_layout(
        margin=dict(l=0, r=0, t=50, b=0),
        height=420,
        geo=dict(showframe=False, showcoastlines=True, landcolor='#95a5a6'),
        coloraxis_colorbar_title='差（%）'
    )
    fig.show()
print('5 maps (最適 vs 2番目 の差 %) done.')

5 maps (最適 vs 2番目 の差 %) done.


### まとめ: 最適 vs 2番目 の差（%）— 5シナリオ一覧

上記5枚の地図を1つの図にまとめた版（3×2、右下は未使用）。

In [9]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

scale_gap = [[0, '#ecf0f1'], [1, '#2c3e50']]
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=order,
    specs=[[{"type": "geo"}, {"type": "geo"}], [{"type": "geo"}, {"type": "geo"}], [{"type": "geo"}, {"type": "geo"}]],
    vertical_spacing=0.06, horizontal_spacing=0.02,
)
for idx, s in enumerate(order):
    d = gap_all[gap_all['scenario'] == s].dropna(subset=['ISO3'])
    row, col = idx // 2 + 1, idx % 2 + 1
    fig.add_trace(
        go.Choropleth(
            locations=d['ISO3'],
            z=d['pct_diff'],
            locationmode='ISO-3',
            colorscale=scale_gap,
            zmin=0, zmax=20,
            hovertext=[f"{a}<br>{v:.1f}%" for a, v in zip(d['Area'], d['pct_diff'])],
            hoverinfo='text',
            showscale=(idx == 0),
            colorbar=dict(title='差（%）', len=0.25, y=0.98) if idx == 0 else None,
        ),
        row=row, col=col,
    )
for r in range(1, 4):
    for c in range(1, 3):
        fig.update_geos(showframe=False, showcoastlines=True, landcolor='#95a5a6', row=r, col=c)
fig.update_layout(
    height=900,
    margin=dict(l=0, r=0, t=60, b=0),
    title_text='最適 vs 2番目 の差（%）— 5シナリオ一覧 [0–20% グラデーション]',
)
fig.show()

In [10]:
color_map = {'Maize (corn)': '#e74c3c', 'Rice': '#27ae60', 'Wheat': '#3498db'}
figs = []
for s in order:
    d = all_opt[all_opt['scenario'] == s].dropna(subset=['ISO3'])
    fig = px.choropleth(
        d,
        locations='ISO3',
        locationmode='ISO-3',
        color='optimal_crop',
        color_discrete_map=color_map,
        hover_name='Area',
        hover_data={'optimal_crop': True, 'effective_kcal': ':,.0f', 'kcal_per_ha': ':,.0f', 'ISO3': False},
        title=f'最適作物（供給係数 コーン0.85, 米0.8, 小麦0.86） — {s}  ({len(d)}か国)'
    )
    fig.update_layout(
        margin=dict(l=0, r=0, t=50, b=0),
        height=420,
        geo=dict(showframe=False, showcoastlines=True, landcolor='#95a5a6'),
        legend_title='最適作物'
    )
    fig.show()
    figs.append(fig)
print('5 maps done.')

5 maps done.


### まとめ: 最適作物マップ — 5シナリオ一覧

上記5枚の最適作物地図を1つの図にまとめた版（右上にダミー白紙、1枚・2枚・2枚の並び）。

In [11]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

color_map = {'Maize (corn)': '#e74c3c', 'Rice': '#27ae60', 'Wheat': '#3498db'}
crop_to_num = {'Maize (corn)': 0, 'Rice': 1, 'Wheat': 2}
scale_disc = [[0, '#e74c3c'], [0.33, '#e74c3c'], [0.33, '#27ae60'], [0.67, '#27ae60'], [0.67, '#3498db'], [1, '#3498db']]

# 空白を None にするとタイトルがずれるため、6枠すべて geo にして右上は空の Choropleth を置く
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[order[0], "", order[1], order[2], order[3], order[4]],
    specs=[[{"type": "geo"}, {"type": "geo"}], [{"type": "geo"}, {"type": "geo"}], [{"type": "geo"}, {"type": "geo"}]],
    vertical_spacing=0.06, horizontal_spacing=0.02,
)
for idx, s in enumerate(order):
    d = all_opt[all_opt['scenario'] == s].dropna(subset=['ISO3']).copy()
    d['z'] = d['optimal_crop'].map(crop_to_num)
    row, col = [1, 2, 2, 3, 3][idx], [1, 1, 2, 1, 2][idx]
    fig.add_trace(
        go.Choropleth(
            locations=d['ISO3'],
            z=d['z'],
            locationmode='ISO-3',
            colorscale=scale_disc,
            zmin=0, zmax=2,
            hovertext=[f"{a}<br>{c}" for a, c in zip(d['Area'], d['optimal_crop'])],
            hoverinfo='text',
            showscale=(idx == 0),
            colorbar=dict(title='最適作物', len=0.25, y=0.98, tickvals=[0.17, 0.5, 0.83], ticktext=['Maize', 'Rice', 'Wheat']) if idx == 0 else None,
        ),
        row=row, col=col,
    )
# 右上 (1,2): 白紙用（何もプロットしない Scattergeo）
fig.add_trace(go.Scattergeo(lat=[], lon=[], hoverinfo='skip'), row=1, col=2)
for r in range(1, 4):
    for c in range(1, 3):
        fig.update_geos(showframe=False, showcoastlines=True, landcolor='#95a5a6', row=r, col=c)
fig.update_layout(
    height=900,
    margin=dict(l=0, r=0, t=60, b=0),
    title_text='最適作物（供給係数 コーン0.85, 米0.8, 小麦0.86）— 5シナリオ一覧',
)
fig.show()